# Entrega 2 · QLoRA para formular LP/MILP

**Orden de revisión:** datos → resultados medidos → demostración del mismo caso → reproducción del ajuste. Requiere GPU T4 para inferencia. El benchmark oficial tiene 30 casos; el ejemplo oftalmológico de Entrega 1 se informa por separado. Las soluciones de referencia nunca se descargan en la sección de generación.

La media **0.5884** del baseline procede del CSV oficial congelado, no de una nueva ejecución. La demo al final hace **dos inferencias nuevas** y las guarda por separado. Los puntajes que muestra la tabla corresponden a los 31 juicios históricos/finales archivados, no a esas nuevas inferencias.


## 1. Instalación y archivos públicos

In [ ]:
%pip -q install transformers==4.51.3 peft==0.15.2 accelerate==1.6.0 bitsandbytes==0.45.5 datasets==3.5.0 pandas matplotlib


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv, hashlib, io, json
import pandas as pd

ROOT = Path('/content/deliverable_2')
ROOT.mkdir(exist_ok=True)
RAW = 'https://raw.githubusercontent.com/Jose21531/GenAI-Optimization/main/Deliverable%202/'
FILES = [
    'data/train_1050_user_only.jsonl',
    'data/benchmark_30_generation_only.jsonl',
    'data/caso_entrega1_generation_only.jsonl',
    'data/SHA256SUMS.generated.json',
    'outputs/baseline_31_respuestas_y_fom5.csv',
    'outputs/qlora_31_respuestas_y_fom5.csv',
]
for name in FILES:
    target = ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(urlopen(RAW + name).read())
expected = json.loads((ROOT/'data/SHA256SUMS.generated.json').read_text(encoding='utf-8'))
for name, digest in expected.items():
    actual = hashlib.sha256((ROOT/'data'/name).read_bytes()).hexdigest()
    assert actual == digest, name

def read_jsonl(name):
    return [json.loads(line) for line in (ROOT/'data'/name).read_text(encoding='utf-8').splitlines()]

train = read_jsonl('train_1050_user_only.jsonl')
benchmark = read_jsonl('benchmark_30_generation_only.jsonl')
case = read_jsonl('caso_entrega1_generation_only.jsonl')[0]
assert len(train) == 1050 and len(benchmark) == 30 and case['id'] == 'diagnostic_oftalmo'
assert all('reference_model' not in item and 'requirements' not in item for item in benchmark)
assert all([m['role'] for m in item['messages']] == ['user','assistant'] for item in train)
print('Datos: 1050 ejemplos de entrenamiento, 30 entradas de benchmark y 1 caso diagnóstico.')
print('Diversidad:', len({x['family_id'] for x in train}), 'familias;',
      sum(x['model_type']=='LP' for x in train), 'LP y',
      sum(x['model_type']=='MILP' for x in train), 'MILP.')


## 2. Resultados oficiales y comparación pareada

Cada CSV contiene la respuesta completa del SLM y el juicio FOM-5 v2. El baseline se recuperó de `resultados_fom5_local.csv` y QLoRA se generó después del ajuste final. El juez Qwen3-14B se aplicó **después** de generar, con la referencia reservada al evaluador. El intervalo bootstrap se calcula aquí sobre diferencias pareadas; no lo entrega automáticamente el juez.


In [ ]:
base = pd.read_csv(ROOT/'outputs/baseline_31_respuestas_y_fom5.csv')
adapt = pd.read_csv(ROOT/'outputs/qlora_31_respuestas_y_fom5.csv')
b = base.query('split == "benchmark"').set_index('id').sort_index()
a = adapt.query('split == "benchmark"').set_index('id').sort_index()
assert len(b) == len(a) == 30 and b.index.equals(a.index)
paired = pd.DataFrame({'baseline': b.score_fom5_v2, 'QLoRA': a.score_fom5_v2})
paired['cambio'] = paired.QLoRA - paired.baseline
counts = {'mejoran': int((paired.cambio > 0).sum()),
          'sin cambio': int((paired.cambio == 0).sum()),
          'empeoran': int((paired.cambio < 0).sum())}
assert counts == {'mejoran': 14, 'sin cambio': 8, 'empeoran': 8}
assert abs(paired.baseline.mean() - .5884) < 1e-10
assert abs(paired.QLoRA.mean() - .8023666666666667) < 1e-10
print(f"Media 30: {paired.baseline.mean():.4f} → {paired.QLoRA.mean():.4f}/5; cambio {paired.cambio.mean():+.4f}")
print('Por caso:', counts)
import numpy as np
rng = np.random.default_rng(42)
means = rng.choice(paired.cambio.to_numpy(), size=(10000, 30), replace=True).mean(axis=1)
print('IC bootstrap 95% del cambio pareado:', np.quantile(means, [.025, .975]).round(4))
print('Caso Entrega 1, aparte: baseline determinista',
      float(base.query('id == "diagnostic_oftalmo"').score_fom5_v2.iloc[0]),
      '→ QLoRA', float(adapt.query('id == "diagnostic_oftalmo"').score_fom5_v2.iloc[0]))
display(paired.sort_values('cambio', ascending=False))


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(['Baseline','QLoRA'], [paired.baseline.mean(), paired.QLoRA.mean()], color=['#74899b','#087f8c'])
axes[0].set_ylim(0, 5); axes[0].set_ylabel('FOM-5 v2 / 5'); axes[0].set_title('Media de 30 casos')
axes[1].bar(counts.keys(), counts.values(), color=['#087f8c','#74899b','#b85b49'])
axes[1].set_ylim(0, 30); axes[1].set_ylabel('Casos'); axes[1].set_title('Cambio por problema')
plt.tight_layout(); plt.show()


## 3. Demostración en vivo: el mismo enunciado de Entrega 1

El adapter final se descarga del repositorio con verificación SHA-256; no hace falta la cuenta ni el Drive del equipo. La primera celda carga el modelo y sus pesos; la siguiente ejecuta un **baseline nuevo** con el adapter apagado y luego el modelo ajustado con igual entrada, decodificación determinista y máximo de 1200 tokens. Los dos textos se guardan en `/content/deliverable_2/outputs/live_demo.json`. La tabla previa conserva los puntajes oficiales congelados y no reetiqueta esta ejecución.


In [ ]:
from zipfile import ZipFile
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

assert torch.cuda.is_available(), 'Selecciona Entorno de ejecución → Cambiar tipo → T4 GPU.'
archive = ROOT/'model/adapter_qlora_qwen25_final.zip'
archive.parent.mkdir(parents=True, exist_ok=True)
if not archive.exists():
    archive.write_bytes(urlopen(RAW+'model/adapter_qlora_qwen25_final.zip').read())
assert hashlib.sha256(archive.read_bytes()).hexdigest() == '68583e9dffa6647a5c0b89a74168363264daf246e4337ef1ae0c644243391151'
ADAPTER = ROOT/'model/adapter'
ADAPTER.mkdir(exist_ok=True)
with ZipFile(archive) as zipped:
    assert set(zipped.namelist()) == {'adapter_config.json','adapter_model.safetensors'}
    zipped.extractall(ADAPTER)
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
REVISION = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
set_seed(42)
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL, revision=REVISION, torch_dtype=torch.float16, device_map={'': 0})
model = PeftModel.from_pretrained(base_model, ADAPTER, is_trainable=False)
model.eval()
print('T4 y adapter público preparados. Inicia la grabación antes de la siguiente celda.')


In [ ]:
import time
print('MISMA ENTRADA · CENTROS OFTALMOLÓGICOS (Entrega 1)')
print(case['problem'])
prompt = tokenizer.apply_chat_template([{'role':'user','content':case['problem']}],
                                       tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(model.device)

def generate_now():
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        tokens = model.generate(**inputs, max_new_tokens=1200, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    new = tokens[0, inputs['input_ids'].shape[1]:]
    return {'candidate': tokenizer.decode(new, skip_special_tokens=True).strip(),
            'output_tokens': int(new.numel()), 'latency_sec': round(time.perf_counter()-started, 3)}

print('Generando baseline directo ahora...', flush=True)
with model.disable_adapter():
    live_base = generate_now()
print('Baseline listo:', live_base['output_tokens'], 'tokens', flush=True)
print('Generando QLoRA ahora...', flush=True)
live_qlora = generate_now()
print('QLoRA listo:', live_qlora['output_tokens'], 'tokens', flush=True)

demo_path = ROOT / 'outputs/live_demo.json'
demo_path.parent.mkdir(parents=True, exist_ok=True)
live_demo = {'id': case['id'], 'problem': case['problem'],
             'problem_sha256': hashlib.sha256(case['problem'].encode()).hexdigest(),
             'model_revision': REVISION, 'prompt': 'user_only', 'do_sample': False,
             'max_new_tokens': 1200, 'baseline_new_run': live_base,
             'qlora_new_run': live_qlora}
demo_path.write_text(json.dumps(live_demo, ensure_ascii=False, indent=2), encoding='utf-8')
print('Dos respuestas nuevas guardadas en:', demo_path)


### Respuestas de la demostración, lado a lado

In [ ]:
from IPython.display import HTML, display
from html import escape
cards = []
for label, run, color in [('Baseline directo · nueva ejecución', live_base, '#74899b'),
                          ('QLoRA · nueva ejecución', live_qlora, '#087f8c')]:
    cards.append(f'''<div style="flex:1;min-width:290px;border:2px solid {color};border-radius:8px;padding:12px">
      <h3 style="color:{color};margin-top:0">{label}</h3>
      <p>{run['output_tokens']} tokens · {run['latency_sec']:.1f} s</p>
      <pre style="white-space:pre-wrap;max-height:540px;overflow:auto;font-size:12px">{escape(run['candidate'])}</pre></div>''')
display(HTML('<div style="display:flex;gap:12px">' + ''.join(cards) + '</div>'))
print('Puntajes del juicio archivado (no de esta ejecución): 0.650 → 3.850 / 5.')
print('Advertencia matemática: QLoRA aún omite p_j en el objetivo del caso oftalmológico.')


## 4. Reproducir el ajuste completo (opcional, ~16 min en T4)

Esta celda vuelve a entrenar desde el modelo base con las **1050** instancias descargadas del repositorio, una época y los hiperparámetros congelados. Ejecútala solo si quieres repetir el experimento: consume GPU y crea una corrida nueva en el entorno de Colab. La corrida medida original, sus manifiestos y el evaluador autoritativo se preservan en `evidence/` y `src/`. Una corrida nueva requiere generar 31 respuestas y evaluarlas de nuevo; no se le deben asignar los puntajes archivados.


In [ ]:
RUN_FULL_TRAINING = False  # Cambiar a True para una reproducción nueva.
if RUN_FULL_TRAINING:
    import torch
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                              Trainer, TrainingArguments, set_seed)
    from datasets import Dataset
    assert torch.cuda.is_available()
    set_seed(42)
    model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
    revision = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
    tok = AutoTokenizer.from_pretrained(model_id, revision=revision)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    def tokenize_row(row):
        prefix = tok.apply_chat_template(row['messages'][:1], tokenize=False, add_generation_prompt=True)
        full = tok.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)
        prefix_ids = tok(prefix, add_special_tokens=False)['input_ids']
        ids = tok(full, add_special_tokens=False)['input_ids']
        assert ids[:len(prefix_ids)] == prefix_ids and len(ids) <= 1024
        labels = [-100]*len(prefix_ids) + ids[len(prefix_ids):]
        assert any(value != -100 for value in labels)
        return {'input_ids': ids, 'attention_mask': [1]*len(ids), 'labels': labels}
    records = [tokenize_row(row) for row in train]
    dataset = Dataset.from_list(records)
    def collate(batch):
        width = max(len(row['input_ids']) for row in batch)
        return {name: torch.tensor([row[name] + [fill]*(width-len(row[name])) for row in batch])
                for name, fill in [('input_ids', tok.pad_token_id), ('attention_mask', 0), ('labels', -100)]}
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision,
        quantization_config=quant, torch_dtype=torch.float16, device_map={'': 0}, attn_implementation='sdpa')
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=.05,
        bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
    fresh = ROOT/'reproduction'
    args = TrainingArguments(output_dir=str(fresh/'checkpoints'), num_train_epochs=1,
        per_device_train_batch_size=1, gradient_accumulation_steps=8, learning_rate=2e-4,
        warmup_ratio=.05, lr_scheduler_type='cosine', fp16=True, bf16=False,
        optim='paged_adamw_8bit', logging_steps=5, save_strategy='steps',
        save_steps=25, save_total_limit=3, report_to='none', remove_unused_columns=False,
        seed=42, data_seed=42, dataloader_num_workers=0, label_names=['labels'])
    trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=collate)
    trainer.train()
    (fresh/'final_adapter').mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(fresh/'final_adapter')
    tok.save_pretrained(fresh/'final_adapter')
    print('Adapter nuevo:', fresh/'final_adapter')


## 5. Auditoría y evaluación posterior

En `evidence/` están el CSV original del baseline, la generación final sin referencias, el CSV completo del juez, manifiestos y el notebook del **evaluador FOM-5 v2 autoritativo**. `src/` conserva los notebooks de entrenamiento, generación y evaluación usados en Colab. El script `data/build_datasets.py` reconstruye los 30 y 1050 desde fuentes congeladas; no afirma recrear el proceso original de autoría sintética. `src/build_outputs.py` reconstruye los dos CSV públicos sin volver a generar ni a juzgar. El IC bootstrap de la celda 2 remuestrea casos emparejados con semilla 42 y 10 000 réplicas; no mide variación entre entrenamientos o jueces.
